In [3]:
from statsmodels.formula.api import ols
import pandas as pd
import numpy as np

Let's run the following regression model to study how sales are associated with different factors. 

 $\text{Sales}=\beta_0+ \beta_1 \text{TV} +\beta_2 \text{Radio}+ \beta_3 \text{Newspaper}+ \epsilon$

Data source: [Advertising Dataset](https://www.kaggle.com/datasets/tawfikelmetwally/advertising-dataset)

* TV: Advertisements on TV in 1000 dollars
* Radio:  Advertisements on Radio in 1000 dollars
* Newspaper:  Advertisements on Newspaper in 1000 dollars
* Sales:  Sales Revenue in M dollars

In [4]:
Sales = pd.read_csv("advertising.csv")
Sales

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9
...,...,...,...,...
195,38.2,3.7,13.8,7.6
196,94.2,4.9,8.1,14.0
197,177.0,9.3,6.4,14.8
198,283.6,42.0,66.2,25.5


In [6]:
#Let's try regressing Sales on the other three columns.
model = ols("Sales ~ TV+Radio+Newspaper", Sales).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  Sales   R-squared:                       0.903
Model:                            OLS   Adj. R-squared:                  0.901
Method:                 Least Squares   F-statistic:                     605.4
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           8.13e-99
Time:                        03:26:14   Log-Likelihood:                -383.34
No. Observations:                 200   AIC:                             774.7
Df Residuals:                     196   BIC:                             787.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.6251      0.308     15.041      0.000       4.019       5.232
TV             0.0544      0.001     39.592      0.000       0.052       0.057
Radio          0.1070      0.008     12.604      0.000       0.090       0.124
Newspaper      0.0003      0.006      0.058      0.954      -0.011       0.012
==============================================================================
Omnibus:                       16.081   Durbin-Watson:                   2.251
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               27.655
Skew:                          -0.431   Prob(JB):                     9.88e-07
Kurtosis:                       4.605   Cond. No.                         454.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

We have a ton of information here;. Let's first look at the columns of the second table.

1. **coef**: Estimated coefficients (`model.params`). These are our $\hat\beta_i$ values. Each of these indicates the effect on Sales when the respective budget increases ($\hat\beta_1$, $\hat\beta_2$, and $\hat\beta_3$), or when no advertisement budget is provided at all ($\hat\beta_0$).

2. **std err**: Standard errors (`model.bse`). Standard errors of each of the estimated coefficients. Remember that these tell us about the uncertainty or reliability of the model.

3. **t**: $t$-statistics (`model.tvalues`). The $t$-statistic is the ratio between the coefficient value and its standard error. These can be used to assess statistical signifiance; a higher $t$-statistic indicates a larger difference from 0 (the null hypothesis), while a lower value indicates the opposite. 

4. **P> |t|**: $p$-values (`model.pvalues`). The infamous $p$-values translate the $t$-statistics to probabilities, specifically that estimated coefficient could have been obtained under the null hypothesis. A small $p$-value (typically $< 0.05$) typically indicates that a predictor is statistically significant. 

5. **[0.025** and **0.975]**: Confidence intervals (`model.conf_int(alpha)`). These are computed from the coefficient estimate and errors, and they express the range of values that contains the true parameters with the given confidence level ($95\%$). The *width* of the interval is related to the estimate uncertainty (error), while its closeness of inclusion of 0 is related to the statistical significance ($t$ and $p$).

In [8]:
# Based on our analysis above, we can see that Newspaper is not a statistically significant predictor for Sales. 
# Let's fit a second model with it removed.
model2 = ols("Sales ~ TV+Radio", Sales).fit()
model2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  Sales   R-squared:                       0.903
Model:                            OLS   Adj. R-squared:                  0.902
Method:                 Least Squares   F-statistic:                     912.7
Date:                Fri, 27 Feb 2026   Prob (F-statistic):          2.39e-100
Time:                        03:27:43   Log-Likelihood:                -383.34
No. Observations:                 200   AIC:                             772.7
Df Residuals:                     197   BIC:                             782.6
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.6309      0.290     15.952      0.000       4.058       5.203
TV             0.0544      0.001     39.726      0.000       0.052       0.057
Radio          0.1072      0.008     13.522      0.000       0.092       0.123
==============================================================================
Omnibus:                       16.227   Durbin-Watson:                   2.252
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               27.973
Skew:                          -0.434   Prob(JB):                     8.43e-07
Kurtosis:                       4.613   Cond. No.                         425.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Now we have a simpler model, and we can still understand the effects of the different advertising budgets on overall sales. Let's look at some of the values from the first table:

**R-squared** and **Adj. R-squared** (`.rsquared` and `.rsquared_adj`): We will not worry about the difference between the two here (the former tends to increase with number of predictors, which "rewards" overfitting). We do see that they are relatively unchanged between the two fitted models, which tells us that dropping Newspaper was appropriate.

**F-statistic** and **Prob (F-statistic)** (`.fvalue`, `.f_pvalue`): These are analogous to the $t$ and $p$ values for the coefficients, but they test significance of the model as a whole. The null hypothesis here is that all $\beta_i = 0$, except for the intercept $\beta_0$. We will usually find that the model is significant as long as at least one of the predictor coefficients is also significant.

In [9]:
# As a final task, we can use the model to perform prediction and compare the results with the observations.
Sales.Sales[:10]

0    22.1
1    10.4
2    12.0
3    16.5
4    17.9
5     7.2
6    11.8
7    13.2
8     4.8
9    15.6
Name: Sales, dtype: float64

In [10]:
model2.predict(Sales[:10])

0    21.210784
1    11.265819
2    10.486714
3    17.306207
4    15.632737
5    10.345422
6    11.277021
7    13.276266
8     5.324207
9    15.788436
dtype: float64